# 这里是 -> **[本篇笔记博客](https://blog.algieba12.cn/llm07-rss-notification-react-agent/)**

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install uv
!uv pip install feedparser llama-index requests llama-index-llms-huggingface 
!uv pip install -U transformers peft accelerate bitsandbytes

In [ ]:
!uv pip install --reinstall torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 --system
# PS: 运行完这两个环境安装命令后 需要重启session

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()


In [ ]:
import feedparser

def get_latest_blog_post(rss_url:str)->str:
    """
    用于请求并解析目标博客的 RSS feed，获取最新的一篇博客文章信息。
    当你需要检查博客是否有更新时，必须首先调用此工具。
    """
    print(f"【calling】 get_latest_blog_post with rss_url:{rss_url}")
    try:
        feed = feedparser.parse(rss_url)
        if feed.entries:
            latest_entry = feed.entries[0]
            print(f"debug: title {latest_entry.title} link:{latest_entry.link}")
            
            return (
                f"Title: {latest_entry.title}\n"
                f"Link: {latest_entry.link}\n\n"
                "【系统强制指令】：请立刻将上面的 Title 与你已知最新标题进行严格比对！\n"
                "情况 A：如果 Title 与已知标题【完全相同】，请直接输出 Answer: 无更新，结束任务。\n"
                "情况 B：如果 Title 与已知标题【不一致】，你绝对不能直接输出 Answer！你必须严格按以下格式输出，以调用通知工具：\n\n"
                "Thought: 标题不一致，我必须立刻调用通知工具。\n"
                "Action: send_all_notifications\n"
                "Action Input: {\"post_title\": \"完全复制上面的Title\", \"post_link\": \"完全复制上面的Link\"}\n"
            )
            
        return f"No posts found in rss feed {rss_url}"
    except Exception as e:
        return f"Exception: {str(e)} on fetching blog"

In [ ]:
get_latest_blog_post("https://blog.algieba12.cn/atom.xml")

In [ ]:
import requests
def send_email_notification(post_title:str, post_link:str)->str:
    """
    当发现博客有更新时，必须调用此工具发送邮件通知。
    参数 post_title: 新博客的标题
    参数 post_link: 新博客的链接
    """
    print(f"【calling】 send_email_notification with post_title {post_title} link {post_link}")
    target_email= user_secrets.get_secret("TARGET_EMAIL")
    email_api_key = user_secrets.get_secret("EMAIL_API_KEY")
    headers = {
        "Authorization": f"Bearer {email_api_key}",
        "Content-Type": "application/json"
    }
    payload = {
        "from" : "onboarding@resend.dev",
        "to":target_email,
        "subject":f"阿尔的代码屋更新咯：{post_title}",
        "text":f"检测到 阿尔的代码屋 更新了一篇新博客 \n\n 标题：{post_title}]\n 链接: {post_link}"
    }

    try:
        response = requests.post("https://api.resend.com/emails", headers=headers, json=payload)
        if response.status_code == 200:
            return "Email sent successfully via API"
        return f"Email sent failed. {response.text}"
    except Exception as e:
        return f"Exception: {str(e)} on sending email"

    ...

In [ ]:
send_email_notification(post_title="email notification test",post_link="none")

In [ ]:
def send_wechat_notification(post_title:str, post_link:str)->str:
    """
    当发现博客有更新时，必须调用此工具发送微信通知。
    参数 post_title: 新博客的标题
    参数 post_link: 新博客的链接
    """
    print(f"【calling】 send_wechat_notification with title:{post_title} link: {post_link}")
    wechat_api_key = user_secrets.get_secret("WECHAT_API_KEY")
    url = f"https://sctapi.ftqq.com/{wechat_api_key}.send"
    data = {
        "title":f"阿尔的代码屋更新咯：{post_title}",
        "desp":f"检测到 阿尔的代码屋 更新了一篇新博客 \n\n 标题：{post_title}]\n 链接: {post_link}"
    }
    try:
        response = requests.post(url,data=data)
        if response.status_code != 200:
            return f"Message sent failed. {response.text}"
        result = response.json()
        if result.get("code") != 0:
            return f"Failed to send notification. API response: {result.get('message')}"
        return "Message sent successfully via API"
        
    except Exception as e:
        return f"Exception: {str(e)} on sending wechat notification"

In [ ]:
# send_wechat_notification(post_title="wechat notification test",post_link="none")

In [ ]:
from llama_index.core.tools import FunctionTool
def send_all_notifications(post_title: str, post_link: str) -> str:
    """
    当发现博客有更新时，必须调用此工具。调用此工具会自动同时发送邮件和微信通知。
    参数 post_title: 新博客的标题
    参数 post_link: 新博客的链接
    """
    print(f"【Agent】 触发了联合通知工具: {post_title}")
    
    email_res = send_email_notification(post_title, post_link)
    wechat_res = send_wechat_notification(post_title, post_link)
    
    return f"邮件通知结果: {email_res} | 微信通知结果: {wechat_res}"

tool_get_blog = FunctionTool.from_defaults(fn=get_latest_blog_post)
tool_send_all = FunctionTool.from_defaults(fn=send_all_notifications)
# tool_send_email = FunctionTool.from_defaults(fn=send_email_notification)
# tool_send_wechat = FunctionTool.from_defaults(fn=send_wechat_notification)

In [ ]:
rss_link = "https://blog.algieba12.cn/atom.xml"
last_title = ""


In [ ]:
local_model_path = "/kaggle/input/datasets/algieba12/qwen3-8b-unsloth-bnb-4bit/Qwen3-8B-unsloth-bnb-4bit/"

In [ ]:
import torch

from llama_index.core import Settings
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.core.agent.workflow import ReActAgent
from llama_index.core.workflow import Context

Settings.llm = HuggingFaceLLM(
    model_name=local_model_path,
    tokenizer_name=local_model_path,
    context_window=8192,
    max_new_tokens=4096,
    generate_kwargs={
        "do_sample":False, # 关闭采样 直接贪婪解码 不随机 直接用可能性最高的结果
    },
    device_map="auto",
    model_kwargs={
        "dtype":torch.float16,
        "trust_remote_code":True,
    }
)

In [ ]:
agent = ReActAgent(
    name="blog_monitor_agent",
    description="自动监控博客并发送通知",
    system_prompt=(
        "你是一个极其严谨的后台监控机器人，严格遵循 Thought-Action-Observation 循环。\n"
        "你的任务是检查博客更新，并在有更新时发送通知。\n\n"
        "【操作规范与强制要求】\n"
        "步骤 1：你必须首先调用 get_latest_blog_post 工具获取真实数据。\n"
        "步骤 2：仔细阅读 Observation 返回的内容。如果获取到的真实标题与系统已知标题不同，说明有更新。\n"
        "步骤 3：如果有更新，你必须立刻调用 send_all_notifications 工具。\n\n"
        "【 **致命错误警告** 】\n"
        "当调用 send_all_notifications 时，传入的 post_title 和 post_link 参数必须 100% 完全复制自 get_latest_blog_post 返回的真实 Observation 数据！\n"
        "绝对禁止凭空捏造参数！绝对禁止使用诸如“新文章”、“test”、“新链接”之类的占位符！必须原样提取真实文本！\n"
        "只有当两个工具都执行完毕，且通知发送成功后，你才能输出最终的 Answer 结束任务。"
    ),
    tools=[tool_get_blog, tool_send_all], 
    llm=Settings.llm,
    max_iterations=8,  # 允许它有足够的步数进行多轮工具调用
    verbose=True
)

In [ ]:
import re

# 初始状态为空
last_title = ""

async def main():
    global last_title
    
    task_prompt = (
        f"请去检查博客 RSS：'{rss_link}'。目前系统已知最新标题是 '{last_title}'。\n"
        "请严格按以下逻辑执行：\n"
        "1. 立即获取最新的真实博客信息。\n"
        "2. 拿到真实信息后，与已知标题进行比对。\n"
        "3. 若发现标题不一致，必须将刚才获取到的真实标题和真实链接提取出来，准确无误地传给通知工具进行发送。"
    )
    ctx = Context(agent)
    try:
        response = await agent.run(user_msg=task_prompt, ctx=ctx)
        result_text = response.response.content.strip()
        print(f"Agent 执行完毕，返回信息: {result_text}")
        
        import feedparser
        feed = feedparser.parse(rss_link)
        if feed.entries:
            current_actual_title = feed.entries[0].title
            if current_actual_title != last_title:
                last_title = current_actual_title
                print(f"系统内部状态已更新，最新标题为：{last_title}")
                
    except Exception as e:
        print(f"Exception: {str(e)}")

In [ ]:
last_title = ""
await main()

In [ ]:
await main()

In [ ]:
last_title = ""
await main()

In [ ]:
async def test_agent_logic():
    global last_title
    
    print("--- 开始测试：场景 A（模拟无更新） ---")
    # 假设系统记录的标题正是 RSS 里目前真实的最新标题
    # 请将这里的字符串替换为你博客目前真实的最新标题
    last_title = "在 Kaggle 上用 Unsloth 极速微调 Qwen3 - 大模型实战 06 | 阿尔的代码屋" 
    
    task_prompt_a = (
        f"请去检查博客 RSS：'{rss_link}'。目前系统已知最新标题是 '{last_title}'。\n"
        "1. 立即获取最新的真实博客信息。\n"
        "2. 拿到真实信息后，与已知标题进行比对。\n"
        "3. 若发现标题不一致，提取标题和链接传给通知工具。若一致，则直接回复无更新。"
    )
    
    ctx_a = Context(agent)
    response_a = await agent.run(user_msg=task_prompt_a, ctx=ctx_a)
    print(f"场景 A 结果（预期：不调用通知工具）：{response_a.response.content.strip()}\n")

    
    print("--- 开始测试：场景 B（模拟有更新） ---")
    # 故意给系统一个旧的或者错误的标题
    last_title = "这是一篇很老的旧文章标题" 
    
    task_prompt_b = (
        f"请去检查博客 RSS：'{rss_link}'。目前系统已知最新标题是 '{last_title}'。\n"
        "1. 立即获取最新的真实博客信息。\n"
        "2. 拿到真实信息后，与已知标题进行比对。\n"
        "3. 若发现标题不一致，提取标题和链接传给通知工具。若一致，则直接回复无更新。"
    )
    
    ctx_b = Context(agent)
    response_b = await agent.run(user_msg=task_prompt_b, ctx=ctx_b)
    print(f"场景 B 结果（预期：调用通知工具并发送真实标题）：{response_b.response.content.strip()}\n")



In [ ]:
# 运行测试

await test_agent_logic()

In [ ]:
async def stress_test_agent(iterations=5):
    global last_title
    success_count = 0
    
    print(f"开始压力测试，共循环 {iterations} 次...")
    
    for i in range(iterations):
        print(f"\n>>>>> 正在执行第 {i+1} 次测试 <<<<<")
        # 每次都模拟“有更新”的状态，看它能不能稳定提取参数
        last_title = f"随机旧标题_测试轮次_{i}"
        
        task_prompt = (
            f"请去检查博客 RSS：'{rss_link}'。目前系统已知最新标题是 '{last_title}'。\n"
            "获取真实信息并比对，若不一致请立刻调用通知工具并提取参数。"
        )
        
        ctx = Context(agent)
        try:
            # 记录执行过程中的异常
            response = await agent.run(user_msg=task_prompt, ctx=ctx)
            result = response.response.content.strip()
            print(f"第 {i+1} 次返回: {result}")
            
            # 这里可以根据你的实际需求加入断言，比如判断结果中是否包含特定词汇
            success_count += 1
        except Exception as e:
            print(f"第 {i+1} 次崩溃: {e}")
            
    print(f"\n测试完成。成功率: {success_count}/{iterations} ({(success_count/iterations)*100}%)")



In [ ]:
# 执行 5 次压力测试（注意监控 Kaggle 显存）

await stress_test_agent(5)